# Generate Paper Outputs (Kaggle)
Aggregates all results from other Kaggle notebooks into manuscript-ready tables and figures.

**Run this last**, after:
- `train_mfft_base.ipynb` / `train_mfft_tiny.ipynb` / `train_mfft_large.ipynb`
- `train_baselines.ipynb`
- `train_ablation_study.ipynb`
- `train_logo.ipynb`
- `paper_evals.ipynb`

All result JSONs must be uploaded to HF before running this.

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "matplotlib", "seaborn", "python-dotenv"], check=False)
print("Ready.")

In [ ]:
# Cell 2: Setup
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

HF_REPO = "MohsinElis/mfft-checkpoints"
OUT_DIR = Path("/kaggle/working/paper_outputs")
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)
print(f"Output: {OUT_DIR}")

In [ ]:
# Cell 3: Download all result files from HF
from huggingface_hub import hf_hub_download, HfApi
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

api = HfApi(token=hf_token)
files = list(api.list_repo_files(repo_id=HF_REPO, repo_type="model"))

result_files = [f for f in files if f.startswith("results/") and f.endswith(".json")]
print(f"Found {len(result_files)} result files on HF:")

all_results = {}
for rf in tqdm(result_files, desc="Downloading results"):
    try:
        path = hf_hub_download(repo_id=HF_REPO, filename=rf, repo_type="model", token=hf_token)
        with open(path) as f:
            all_results[rf] = json.load(f)
        print(f"  Downloaded: {rf}")
    except Exception as e:
        print(f"  FAILED: {rf} — {e}")

In [ ]:
# Cell 4: Download MFFT training state files for tiny/base/large
mfft_runs = {}
for f in files:
    if "training_state.json" in f and any(v in f for v in ["mfft-tiny", "mfft-base", "mfft-large"]):
        try:
            path = hf_hub_download(repo_id=HF_REPO, filename=f, repo_type="model", token=hf_token)
            with open(path) as fh:
                state = json.load(fh)
            variant = "tiny" if "tiny" in f else "large" if "large" in f else "base"
            mfft_runs[variant] = state
            print(f"  {variant}: best_f1={state.get('best_val_macro_f1', '?')} epoch={state.get('best_model_epoch', '?')}")
        except Exception:
            pass

In [ ]:
# Cell 5: Table 1 — MFFT Variant Comparison
print("\n" + "="*70)
print("TABLE 1: MFFT Variant Comparison")
print("="*70)
rows = []
for v in ["tiny", "base", "large"]:
    if v in mfft_runs:
        s = mfft_runs[v]
        rows.append({
            "Variant": f"MFFT-{v.title()}",
            "Best Epoch": s.get("best_model_epoch", "?"),
            "Val Macro-F1": f"{s.get('best_val_macro_f1', 0):.4f}",
            "Val AUC": f"{s.get('best_val_auc', 0):.4f}",
            "Status": s.get("status", "?"),
        })
if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("No MFFT runs found.")

In [ ]:
# Cell 6: Table 2 — Baseline Comparison
print("\n" + "="*70)
print("TABLE 2: Baseline Comparison")
print("="*70)
if "results/baselines/baselines.json" in all_results:
    baselines = all_results["results/baselines/baselines.json"]
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in baselines])
    df = df.sort_values("test_f1", ascending=False)
    print(df.to_string(index=False))
else:
    print("No baseline results found.")

In [ ]:
# Cell 7: Table 3 — Ablation Study
print("\n" + "="*70)
print("TABLE 3: Ablation Study")
print("="*70)
if "results/ablation_results.json" in all_results:
    ablations = all_results["results/ablation_results.json"]
    df = pd.DataFrame(ablations)
    df = df.sort_values("test_f1", ascending=False)
    print(df[["name", "n_params", "val_best_f1", "test_acc", "test_f1"]].to_string(index=False))
else:
    print("No ablation results found.")

In [ ]:
# Cell 8: Table 4 — LOGO Generalization
print("\n" + "="*70)
print("TABLE 4: Leave-One-Generator-Out (LOGO)")
print("="*70)
if "results/logo_results.json" in all_results:
    logo = all_results["results/logo_results.json"]
    df = pd.DataFrame(logo)
    print(df.to_string(index=False))
    print(f"\nMean held-out accuracy: {df['test_acc'].mean():.4f}")
    print(f"Mean held-out F1:       {df['test_f1'].mean():.4f}")
else:
    print("No LOGO results found.")

In [ ]:
# Cell 9: Table 5 — Paper Evaluations (CIs, Calibration, Robustness)
print("\n" + "="*70)
print("TABLE 5: Paper Evaluations")
print("="*70)
if "results/paper_evals.json" in all_results:
    pe = all_results["results/paper_evals.json"]
    print("\nBootstrap 95% CIs:")
    for metric, vals in pe.get("bootstrap_ci", {}).items():
        print(f"  {metric}: {vals['mean']:.4f} [{vals['lo']:.4f}, {vals['hi']:.4f}]")
    print("\nCalibration:")
    cal = pe.get("calibration", {})
    print(f"  Temperature: {cal.get('temperature', '?')}")
    print(f"  Pre-cal  ECE: {cal.get('pre_ece', '?')} | Brier: {cal.get('pre_brier', '?')}")
    print(f"  Post-cal ECE: {cal.get('post_ece', '?')} | Brier: {cal.get('post_brier', '?')}")
    print("\nRobustness:")
    for r in pe.get("robustness", []):
        print(f"  {r['name']:20s} acc={r['acc']:.4f} f1={r['f1']:.4f}")
else:
    print("No paper eval results found.")

In [ ]:
# Cell 10: Figures
sns.set_theme(style="whitegrid", font_scale=1.1)

# Figure 1: MFFT variant comparison bar chart
if mfft_runs:
    fig, ax = plt.subplots(figsize=(8, 4))
    variants = list(mfft_runs.keys())
    f1s = [mfft_runs[v].get("best_val_macro_f1", 0) for v in variants]
    bars = ax.bar(variants, f1s, color=["#3498db", "#2ecc71", "#e74c3c"][:len(variants)])
    ax.set_ylabel("Best Val Macro-F1")
    ax.set_title("MFFT Variant Comparison")
    for bar, f1 in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{f1:.3f}", ha="center")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "figures/mfft_variant_comparison.png", dpi=150)
    plt.show()
    print("Saved: mfft_variant_comparison.png")

# Figure 2: Baseline comparison
if "results/baselines/baselines.json" in all_results:
    baselines = all_results["results/baselines/baselines.json"]
    fig, ax = plt.subplots(figsize=(10, 5))
    names = [b["name"] for b in baselines]
    f1s = [b["test_f1"] for b in baselines]
    bars = ax.barh(names, f1s, color=sns.color_palette("viridis", len(names)))
    ax.set_xlabel("Test Macro-F1")
    ax.set_title("Baseline Comparison")
    for bar, f1 in zip(bars, f1s):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2, f"{f1:.3f}", va="center")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "figures/baseline_comparison.png", dpi=150)
    plt.show()
    print("Saved: baseline_comparison.png")

# Figure 3: Ablation study
if "results/ablation_results.json" in all_results:
    ablations = all_results["results/ablation_results.json"]
    fig, ax = plt.subplots(figsize=(10, 5))
    names = [a["name"] for a in ablations]
    f1s = [a["test_f1"] for a in ablations]
    bars = ax.barh(names, f1s, color=sns.color_palette("magma", len(names)))
    ax.set_xlabel("Test Macro-F1")
    ax.set_title("Ablation Study")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "figures/ablation_study.png", dpi=150)
    plt.show()
    print("Saved: ablation_study.png")

# Figure 4: LOGO generalization
if "results/logo_results.json" in all_results:
    logo = all_results["results/logo_results.json"]
    fig, ax = plt.subplots(figsize=(10, 5))
    gens = [r["held_out"] for r in logo]
    accs = [r["test_acc"] for r in logo]
    bars = ax.barh(gens, accs, color=sns.color_palette("coolwarm", len(gens)))
    ax.set_xlabel("Test Accuracy")
    ax.set_title("LOGO: Cross-Generator Generalization")
    ax.axvline(x=np.mean(accs), color="red", linestyle="--", label=f"Mean: {np.mean(accs):.3f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "figures/logo_generalization.png", dpi=150)
    plt.show()
    print("Saved: logo_generalization.png")

# Figure 5: Robustness degradation
if "results/paper_evals.json" in all_results:
    pe = all_results["results/paper_evals.json"]
    if "robustness" in pe:
        fig, ax = plt.subplots(figsize=(8, 4))
        names = [r["name"] for r in pe["robustness"]]
        accs = [r["acc"] for r in pe["robustness"]]
        ax.bar(names, accs, color=sns.color_palette("RdYlGn", len(names)))
        ax.set_ylabel("Accuracy")
        ax.set_title("Robustness Under Perturbation")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "figures/robustness.png", dpi=150)
        plt.show()
        print("Saved: robustness.png")

In [ ]:
# Cell 11: Save combined manuscript metrics
manuscript = {
    "mfft_variants": mfft_runs,
    "baselines": all_results.get("results/baselines/baselines.json", []),
    "ablations": all_results.get("results/ablation_results.json", []),
    "logo": all_results.get("results/logo_results.json", []),
    "paper_evals": all_results.get("results/paper_evals.json", {}),
}

with open(OUT_DIR / "manuscript_metrics.json", "w") as f:
    json.dump(manuscript, f, indent=2, default=str)

# Upload all outputs to HF
from huggingface_hub import HfApi
api = HfApi(token=hf_token)

for p in OUT_DIR.rglob("*"):
    if p.is_file():
        rel = p.relative_to(OUT_DIR)
        api.upload_file(
            path_or_fileobj=str(p),
            path_in_repo=f"paper_outputs/{rel}",
            repo_id=HF_REPO, repo_type="model",
        )

print(f"\nAll outputs saved to {OUT_DIR}")
print(f"Uploaded to: https://huggingface.co/{HF_REPO}/tree/main/paper_outputs")